# Character-Level Language Model

A GPT-style character-level language model trained on embedded Shakespeare text.
Follows the Karpathy char-rnn/minGPT tradition: sliding window over a small
corpus, next-character prediction at every position.

Reuses the multi-block transformer from the sorting example, but with a larger
vocabulary (36 chars: a-z, space, newline, punctuation) and a longer context
window.

**CLI equivalent:** `make example-gpt` (2000 epochs, seqLen=64, batch=32)

## Architecture

Same transformer stack as the sorting task, but configured for language modeling:
- Vocabulary: 36 characters (a-z, space, newline, 8 punctuation marks)
- Sequence length: 64 characters
- Embedding dim: 64, 4 heads, head dim 16, 2 blocks

Input: token indices `[seqLen]`. Output: logits `[seqLen * vocabSize]`.
Loss: cross-entropy on all positions (standard LM loss).

In [ ]:
:t mkTransformer

## Model Construction

We can build and inspect the model interactively. The corpus embedding,
tokenization, and training data generation are in the compiled example
(`src/Example/Gpt.idr`).

In [ ]:
:exec do {
  tfm <- mkTransformer {ty = Variable CPU, seqLen=32, dModel=32, numHeads=4, headDim=8, numBlocks=2, vocabSize=36};
  namedTfm <- pure (nameLayer "tfm0" tfm);
  model <- pure (OutputLayer (MkAnyLayer (TransformerState 32 32 4 8 2 36) namedTfm));
  putStrLn ("Model: " ++ show model) }

Training requires corpus tokenization and sliding-window data generation
(defined in `src/Example/Gpt.idr`). The training loop:

```idris
opt <- pure (nativeAdamW 0.001 0.9 0.999 1.0e-8 0.01 1.0)
batchFwd <- pure (transformerForwardBatch namedTfm)
(trained, epochs, loss) <- runTraining
  (\m, d => epochNativeTensorBatch opt d batchFwd allPositionsCE m)
  (gptBatchVect BatchSize) (simpleConfig 2000) model
```

Run via CLI: `make example-gpt --epochs 2000`

## Autoregressive Generation

After training, the model generates text by repeatedly:
1. Forward pass on context window
2. Sample or argmax the last position's logits
3. Append the new token, shift the window

The CLI example generates coherent-ish Shakespeare after 2000 epochs.
At 200 epochs, expect mostly recognizable character patterns but not
real words.

## Scaling Up

For full convergence:
```bash
make example-gpt --epochs 2000 --lr 0.001
```

Expected output after 2000 epochs:
- BPC (bits per character) around 2.0-2.5
- Generated text shows recognizable English words and Shakespeare-like patterns

The corpus is intentionally tiny (1342 chars). A real char-LM would use
megabytes of text, but this demonstrates the architecture end-to-end.

## PyTorch Comparison

```python
# Karpathy's minGPT pattern
model = GPT(vocab_size=36, block_size=64, n_embd=64,
            n_head=4, n_layer=2)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(2000):
    x, y = get_batch(corpus, block_size=64)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, 36), y.view(-1))
    loss.backward()
    optimizer.step()
```

See `pytorch/torch_ref/scripts/gpt.py` for the full reference.

Next: [NTM](ntm.ipynb) — neural networks with external memory.